# Dummy tests about error handling

In [3]:
import sys
import os.path
sys.path.append(os.path.dirname(os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(os.getcwd()))))))

from pylondrina.issues.core import IssueSpec, emit_and_maybe_raise
from pylondrina.errors import SchemaError
from pylondrina.errors import ImportError as PylondrinaImportError

In [2]:
DUMMY_ERR_ISSUES = {
  # warning no fatal
  "TST.WARN.NONFATAL": IssueSpec(
      code="TST.WARN.NONFATAL",
      level="warning",
      fatal=False,
      exception="import",
      message_template="Warn {field}",
      details_keys=("field",),
  ),

  # info no fatal
  "TST.INFO.NONFATAL": IssueSpec(
      code="TST.INFO.NONFATAL",
      level="info",
      fatal=False,
      exception="import",
      message_template="Info ok",
  ),

  # error no fatal (solo strict)
  "TST.ERROR.STRICT_ONLY": IssueSpec(
      code="TST.ERROR.STRICT_ONLY",
      level="error",
      fatal=False,            # clave
      exception="import",
      message_template="Error strict-only {reason}",
      details_keys=("reason",),
  ),

  # error fatal siempre
  "TST.ERROR.FATAL_IMPORT": IssueSpec(
      code="TST.ERROR.FATAL_IMPORT",
      level="error",
      fatal=True,
      exception="import",
      message_template="Fatal import {why}",
      details_keys=("why",),
  ),

  # error fatal schema
  "TST.ERROR.FATAL_SCHEMA": IssueSpec(
      code="TST.ERROR.FATAL_SCHEMA",
      level="error",
      fatal=True,
      exception="schema",
      message_template="Fatal schema {schema_version}",
      details_keys=("schema_version",),
  ),
}

In [4]:
EXC_MAP = {"schema": SchemaError, "import": PylondrinaImportError}

## Test A: No lanza en warning/info (strict True/False)

Cuando los issues no son de nivel "error", sin importar strict, no se levantar errores

In [6]:
# Prueba con issue de nivel warning (no es un error) y en modo estricto
issues = []
issue = emit_and_maybe_raise(
    issues, DUMMY_ERR_ISSUES, "TST.WARN.NONFATAL",
    strict=True, exception_map=EXC_MAP, default_exception=ImportError,
    field="mode"
)
assert issue.code == "TST.WARN.NONFATAL"
assert len(issues) == 1

# Prueba con issue de invel info (no es error) y en modo no estricto
issues = []
issue = emit_and_maybe_raise(
    issues, DUMMY_ERR_ISSUES, "TST.INFO.NONFATAL",
    strict=False, exception_map=EXC_MAP, default_exception=ImportError
)
assert issue.level == "info"
assert len(issues) == 1

## Test B: Error nonfatal: no lanza si strict=False, sí lanza si strict=True

Caso con errores que no son fatales, cuando el issue se emite en modo no estricto no se lanza una Excepción. EN cambio, cuando el issue se emite en modo estricto si se arroja la Excepción.

In [10]:
# strict=False -> NO raise
issues = []
issue = emit_and_maybe_raise(
    issues, DUMMY_ERR_ISSUES, "TST.ERROR.STRICT_ONLY",
    strict=False, exception_map=EXC_MAP, default_exception=PylondrinaImportError,
    reason="x"
)
assert issue.level == "error"
assert len(issues) == 1  # se emitió igual

# strict=True -> raise ImportError
issues = []
try:
    emit_and_maybe_raise(
        issues, DUMMY_ERR_ISSUES, "TST.ERROR.STRICT_ONLY",
        strict=True, exception_map=EXC_MAP, default_exception=PylondrinaImportError,
        reason="x"
    )
    assert False, "Debió lanzar"
except PylondrinaImportError as e:
    assert e.code == "TST.ERROR.STRICT_ONLY"
    assert e.issue is issues[-1]
    assert e.issues is not None and len(e.issues) == 1
    assert e.details["reason"] == "x"

## Test C: Fatal=True lanza siempre, independiente de strict

Prueba de que sin importar el modo estricto, si el error es fatal, siempre se va a levantar un error

In [12]:
for strict in (False, True):
    issues = []
    try:
        emit_and_maybe_raise(
            issues, DUMMY_ERR_ISSUES, "TST.ERROR.FATAL_IMPORT",
            strict=strict, exception_map=EXC_MAP, default_exception=PylondrinaImportError,
            why="boom"
        )
        assert False, "Debió lanzar siempre"
    except PylondrinaImportError as e:
        assert e.code == "TST.ERROR.FATAL_IMPORT"
        assert e.details["why"] == "boom"
        assert e.issue is issues[-1]
        assert len(e.issues) == len(issues) == 1

## Test D: Tipo de excepción según spec.exception

Prueba de que se prioriza el tipo de error especificado en el exception_map

In [16]:
issues = []
try:
    emit_and_maybe_raise(
        issues, DUMMY_ERR_ISSUES, "TST.ERROR.FATAL_SCHEMA",
        strict=False, exception_map=EXC_MAP, default_exception=PylondrinaImportError,
        schema_version="1.1"
    )
    assert False
except SchemaError as e:
    assert e.code == "TST.ERROR.FATAL_SCHEMA"
    assert e.details["schema_version"] == "1.1"

## Test E: La issue gatillante queda registrada antes de lanzar

Prueba de que no se pierda evidencia.

In [13]:
issues = []
try:
    emit_and_maybe_raise(
        issues, DUMMY_ERR_ISSUES, "TST.ERROR.FATAL_IMPORT",
        strict=False, exception_map=EXC_MAP, default_exception=PylondrinaImportError,
        why="boom"
    )
except PylondrinaImportError:
    pass

assert len(issues) == 1
assert issues[0].code == "TST.ERROR.FATAL_IMPORT"

## Test F: Issues acumuladas previas se adjuntan al error

Prueba de que se guarda la evidencia o rastro de Issues previas cuando se arroja una Excepción

In [19]:
issues = []
emit_and_maybe_raise(
    issues, DUMMY_ERR_ISSUES, "TST.WARN.NONFATAL",
    strict=False, exception_map=EXC_MAP, default_exception=PylondrinaImportError,
    field="mode"
)
try:
    emit_and_maybe_raise(
        issues, DUMMY_ERR_ISSUES, "TST.ERROR.FATAL_IMPORT",
        strict=False, exception_map=EXC_MAP, default_exception=PylondrinaImportError,
        why="boom"
    )
    assert False
except PylondrinaImportError as e:
    assert len(e.issues) == 2      # warning + fatal
    assert e.issues[0].code == "TST.WARN.NONFATAL"
    assert e.issues[1].code == "TST.ERROR.FATAL_IMPORT"